In [1]:
source(here::here('MG3.functions1.R'))

# Checking each HMM result across the genomes

In [2]:
queryTxt="SELECT DISTINCT ensembl_gene_id FROM positively_ided_using_context"
geneIDs=dbGetQuery(mydbRO, queryTxt)[[1]]

In [3]:
# create dir
comToBash=paste0("mkdir -p ",OutputDir,"step7.everyHMMsearch5MB/slurm_logs")
system(comToBash, intern = FALSE, wait = TRUE)

sbatchscript="~/data/lambda/notebooks/Rscript.wrapper.lambdaenv.sh ~/data/lambda/notebooks/MG3.Step7.RunHMMallSpecies.5MB.R"
swarmlines=paste(sbatchscript, geneIDs) 

swarmparams=paste0("#SWARM -t 1 -g 20 --time=3:00:00 --gres=lscratch:20 --partition=quick --logdir ", path.expand(OutputDir),"step7.everyHMMsearch5MB/slurm_logs/")
maxchunk=1000
timestamp=getTimeStamp()
processed=0
i=0
swarmFiles=NULL
while (processed<length(swarmlines)){
    upperlimit=processed+min(maxchunk, length(swarmlines)-processed)
    message(paste(processed+1, upperlimit))
    fname=paste0(OutputDir,"step7.everyHMMsearch5MB/runningGeneID.",i,"-", timestamp,".swarm")
    writeLines(text = c(swarmparams, swarmlines[(processed+1):upperlimit]), 
               con=fname)
    swarmFiles=addToVec(swarmFiles, fname)
    processed=upperlimit
    i=i+1
}

1 1000

1001 2000

2001 3000

3001 4000

4001 5000

5001 6000

6001 7000

7001 8000

8001 9000

9001 9608



## Scheduling the computations

In [4]:
#Run once
startedBool=rep(FALSE, length(swarmFiles))
while (!all(startedBool)){
    for (i in 1:length(swarmFiles)){
        if (!startedBool[i]){
            comToBash=paste0("swarm ", swarmFiles[i])
            message(comToBash)
            retval=system(comToBash, wait = TRUE)
            if (retval == 0){
                startedBool[i]=TRUE
                message("\t Success")
            }else{
                message("\t Nope")
                Sys.sleep(2)
            }
        }
    }
    if (!all(startedBool)){
        message(sum(!startedBool) ,"left")
        Sys.sleep(120)
    }
}
system('echo 1 > /home/frimanav/data/lambda/output/donestep7.everyHMMsearchschedulting', intern = TRUE, wait = TRUE)


swarm ~/data/lambda/output/step7.everyHMMsearch5MB/runningGeneID.0-2026-01-29_10.37.16.144324.swarm

	 Success

swarm ~/data/lambda/output/step7.everyHMMsearch5MB/runningGeneID.1-2026-01-29_10.37.16.144324.swarm

	 Success

swarm ~/data/lambda/output/step7.everyHMMsearch5MB/runningGeneID.2-2026-01-29_10.37.16.144324.swarm

	 Success

swarm ~/data/lambda/output/step7.everyHMMsearch5MB/runningGeneID.3-2026-01-29_10.37.16.144324.swarm

	 Nope

swarm ~/data/lambda/output/step7.everyHMMsearch5MB/runningGeneID.4-2026-01-29_10.37.16.144324.swarm

	 Nope

swarm ~/data/lambda/output/step7.everyHMMsearch5MB/runningGeneID.5-2026-01-29_10.37.16.144324.swarm

	 Nope

swarm ~/data/lambda/output/step7.everyHMMsearch5MB/runningGeneID.6-2026-01-29_10.37.16.144324.swarm

	 Nope

swarm ~/data/lambda/output/step7.everyHMMsearch5MB/runningGeneID.7-2026-01-29_10.37.16.144324.swarm

	 Nope

swarm ~/data/lambda/output/step7.everyHMMsearch5MB/runningGeneID.8-2026-01-29_10.37.16.144324.swarm

	 Nope

swarm ~/da

character(0)

## Checking completion

In [20]:
compelted_genes=list.files(paste0(OutputDir,"step7.everyHMMsearch5MB/"), 
                     pattern = "ENSG.*RDS", full.names = FALSE)
compelted_genes=gsub(pattern = ".RDS", replacement = "", x = compelted_genes, fixed = TRUE)
problem_genes=setdiff(geneIDs, compelted_genes)

In [21]:
length(problem_genes)

[1] 0

In [22]:
problem_genes

character(0)

### Rerunning the problem genes using more dedicated script

In [11]:
queryTxt="SELECT DISTINCT accession FROM GTFs"
accessions=dbGetQuery(mydbRO, queryTxt)[[1]]

In [16]:
problem_cartesians <- expand.grid(x = problem_genes, y = accessions)
problem_cartesians=problem_cartesians %>% mutate(text=paste(x, y))
problem_cartesians=problem_cartesians$text

In [18]:
# create dir
comToBash=paste0("mkdir -p ",OutputDir,"step7.everyHMMsearch5MB/PARALLEL/slurm_logs")
system(comToBash, intern = FALSE, wait = TRUE)


sbatchscript="~/data/lambda/notebooks/Rscript.wrapper.lambdaenv.sh ~/data/lambda/notebooks/MG3.Step7.RunHMMallSpecies-PARALLEL.5MB.R"
swarmlines=paste(sbatchscript, problem_cartesians) 

swarmparams=paste0("#SWARM -t 1 -g 20 --time=3:00:00 --gres=lscratch:20 --partition=quick --logdir ", path.expand(OutputDir),"step7.everyHMMsearch5MB/PARALLEL/slurm_logs/")
maxchunk=1000
timestamp=getTimeStamp()
processed=0
i=0
swarmFiles=NULL
while (processed<length(swarmlines)){
    upperlimit=processed+min(maxchunk, length(swarmlines)-processed)
    message(paste(processed+1, upperlimit))
    fname=paste0(OutputDir,"step7.everyHMMsearch5MB/PARALLEL/runningGeneID.",i,"-", timestamp,".swarm")
    writeLines(text = c(swarmparams, swarmlines[(processed+1):upperlimit]), 
               con=fname)
    swarmFiles=addToVec(swarmFiles, fname)
    message("Run from bash: ",fname)
    processed=upperlimit
    i=i+1
}

1 1000

Run from bash: ~/data/lambda/output/step7.everyHMMsearch5MB/PARALLEL/runningGeneID.0-2026-01-29_13.32.59.848371.swarm

1001 2000

Run from bash: ~/data/lambda/output/step7.everyHMMsearch5MB/PARALLEL/runningGeneID.1-2026-01-29_13.32.59.848371.swarm

2001 2013

Run from bash: ~/data/lambda/output/step7.everyHMMsearch5MB/PARALLEL/runningGeneID.2-2026-01-29_13.32.59.848371.swarm



### Collecting the results

In [19]:
sbatchscript="~/data/lambda/notebooks/Rscript.wrapper.lambdaenv.sh ~/data/lambda/notebooks/MG3.Step7.RunHMMallSpecies-PAR-COLLECT.5MB.R"
swarmlines=paste(sbatchscript, problem_genes) 
swarmparams=paste0("#SWARM -t 1 -g 20 --time=3:00:00 --gres=lscratch:20 --partition=quick --logdir ", path.expand(OutputDir),"step7.everyHMMsearch5MB/PARALLEL/slurm_logs/")
maxchunk=1000
timestamp=getTimeStamp()
processed=0
i=0
swarmFiles=NULL
while (processed<length(swarmlines)){
    upperlimit=processed+min(maxchunk, length(swarmlines)-processed)
    message(paste(processed+1, upperlimit))
    fname=paste0(OutputDir,"step7.everyHMMsearch5MB/PARALLEL/collectingGeneID.",i,"-", timestamp,".swarm")
    writeLines(text = c(swarmparams, swarmlines[(processed+1):upperlimit]), 
               con=fname)
    swarmFiles=addToVec(swarmFiles, fname)
    message("Run from bash: ",fname)
    processed=upperlimit
    i=i+1
}

1 11

Run from bash: ~/data/lambda/output/step7.everyHMMsearch5MB/PARALLEL/collectingGeneID.0-2026-01-29_14.09.07.73058.swarm



## Processing the results once the problem genes are resolved

In [24]:
# loading the search results
geneNamesDF_list=list()
geneFiles=list.files(paste0(OutputDir,"step7.everyHMMsearch5MB/"), 
                     pattern = "ENSG.*RDS", full.names = TRUE)
i=0
for (geneFile in geneFiles){
    if (i %% 500 ==0 ) message(i, " / ", length(geneFiles))
    geneID=basename(geneFile)
    geneID=str_remove(string = geneID, pattern = "\\.RDS")
    tmpList=readRDS(geneFile)
    if (length(tmpList)==0) next
    tmpDF=data.frame(ensembl_gene_id=geneID, accession=names(tmpList), geneName=unlist(tmpList))
    rownames(tmpDF)<-NULL
    geneNamesDF_list[[geneID]]= tmpDF
    i=i+1
}

0 / 9608

500 / 9608

1000 / 9608

1500 / 9608

2000 / 9608

2500 / 9608

3000 / 9608

3500 / 9608

4000 / 9608

4500 / 9608

5000 / 9608

5500 / 9608

6000 / 9608

6500 / 9608

7000 / 9608

7500 / 9608

8000 / 9608

8500 / 9608

9000 / 9608

9500 / 9608



In [25]:
geneNamesDF <- bind_rows(geneNamesDF_list)

In [27]:
# run ONCE
dbWriteTable(mydbRW, "AllSpeciesFoundGeneNames5MB", geneNamesDF, append=FALSE)

In [28]:
dbExecute(mydbRW, 'CREATE VIEW EutherianGenomes AS SELECT DISTINCT accession AS accession FROM taxonomy WHERE id=="9347" /* EutherianGenomes(accession) */')


[1] 0

In [29]:
dbExecute(mydbRW, 'CREATE INDEX AllSpeciesFoundGeneNames5MBEnsInd ON AllSpeciesFoundGeneNames5MB(ensembl_gene_id, accession)')
dbExecute(mydbRW, 'CREATE INDEX AllSpeciesFoundGeneNames5MBAccInd ON AllSpeciesFoundGeneNames5MB(accession, ensembl_gene_id)')
dbExecute(mydbRW, 'CREATE INDEX AllSpeciesFoundGeneNames5MBGeneInd ON AllSpeciesFoundGeneNames5MB(geneName)')

[1] 0

[1] 0

[1] 0

# Visulizations and basic stats (not part of the pipeline)

In [3]:
queryTxt="SELECT * FROM AllSpeciesFoundGeneNames5MB"
geneNamesDF=dbGetQuery(mydbRO, queryTxt)
tmpDF1=geneNamesDF %>% group_by(accession) %>% summarize(num_genes=n())

In [4]:
mean(tmpDF1$num_genes)

[1] 9312.596

In [16]:
median(tmpDF1$num_genes)

[1] 9402

In [5]:
max(tmpDF1$num_genes)

[1] 9511

In [6]:
length(unique(geneNamesDF$ensembl_gene_id))

[1] 9604

In [7]:
median(tmpDF1$num_genes)

[1] 9402

In [8]:
min(tmpDF1$num_genes)

[1] 5195

In [9]:
tmpDF2=geneNamesDF %>% group_by(ensembl_gene_id) %>% summarize(num_genomes=n())

In [10]:
mean(tmpDF2$num_genomes)

[1] 177.4474

In [11]:
length(unique(geneNamesDF$accession))

[1] 183

In [12]:
max(tmpDF2$num_genomes)

[1] 183

In [13]:
median(tmpDF2$num_genomes)

[1] 180

In [14]:
min(tmpDF2$num_genomes)

[1] 1

In [15]:
tmpDF2[ tmpDF2$num_genomes<3,]

ensembl_gene_id,num_genomes
<chr>,<int>
ENSG00000003096,1
ENSG00000036448,2
ENSG00000086967,2
ENSG00000116039,1
ENSG00000120729,2
ENSG00000127955,2
ENSG00000134571,2
ENSG00000142661,2
ENSG00000144566,2


In [ ]:
tmpDF2[ tmpDF2$ ]